In [1]:

from pathlib import Path
from urllib.request import urlretrieve

import pyspark
from pyspark.sql import SparkSession

from pathlib import Path

In [2]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("zoomcamp-homework")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 02:11:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
#question 1

spark.version


'4.2.0'

In [4]:

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"
file_path = Path("data/raw/yellow/2025/11/yellow_tripdata_2025-11.parquet")

file_path.parent.mkdir(parents=True, exist_ok=True)

if not file_path.exists():
    urlretrieve(url, file_path)

df_yellow = spark.read.parquet(str(file_path))

df_yellow.printSchema()



root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [5]:
#Question 2

output_path = "data/pq/yellow/2025/11"



df_yellow.repartition(4) \
    .write \
    .mode("overwrite") \
    .parquet(output_path)

In [6]:
#Question 3 

df_yellow.createOrReplaceTempView("yellow_trips")

spark.sql("""
    SELECT COUNT(*) AS trip_count
    FROM yellow_trips
    WHERE TO_DATE(tpep_pickup_datetime) = '2025-11-15'
""").show()

[Stage 4:>                                                          (0 + 2) / 2]

+----------+
|trip_count|
+----------+
|    162604|
+----------+



In [7]:
spark.sql("""
    SELECT MAX(
        (unix_timestamp(tpep_dropoff_datetime)
         - unix_timestamp(tpep_pickup_datetime)) / 3600.0
    ) AS longest_trip_hours
    FROM yellow_trips
""").show()

[Stage 7:>                                                          (0 + 2) / 2]

+------------------+
|longest_trip_hours|
+------------------+
|         90.646667|
+------------------+



In [8]:

url_zones = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zones_path = Path("data/raw/taxi_zone_lookup.csv")

zones_path.parent.mkdir(parents=True, exist_ok=True)

if not zones_path.exists():
    urlretrieve(url_zones, zones_path)

df_zones = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(zones_path))
)

df_zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [10]:
df_zones.createOrReplaceTempView("taxi_zones")

spark.sql("""
    SELECT z.Zone, COUNT(*) AS pickup_count
    FROM yellow_trips AS t
    JOIN taxi_zones AS z
      ON t.PULocationID = z.LocationID
    GROUP BY z.Zone
    ORDER BY pickup_count ASC
    limit 10
""").show()

[Stage 17:=============================>                            (1 + 1) / 2]

+--------------------+------------+
|                Zone|pickup_count|
+--------------------+------------+
|Governor's Island...|           1|
|Eltingville/Annad...|           1|
|       Arden Heights|           1|
|       Port Richmond|           3|
|       Rikers Island|           4|
|   Rossville/Woodrow|           4|
|         Great Kills|           4|
| Green-Wood Cemetery|           4|
|         Jamaica Bay|           5|
|         Westerleigh|          12|
+--------------------+------------+

